In [1]:
# 1. 필요한 외부 라이브러리 설치
!pip install yacs lmdbdict

# 2. 가중치 및 테스트 이미지를 저장할 폴더 생성
import os

os.makedirs('data/imagenet_weights', exist_ok=True)
os.makedirs('test_images', exist_ok=True)
print("필수 디렉토리 생성 완료.")


   ---------------------------------------- 2/2 [lmdbdict]

필수 디렉토리 생성 완료.


In [2]:
import urllib.request
import os

# PyTorch 공식 ResNet101 가중치 URL 및 저장 경로 지정
resnet101_url = "https://download.pytorch.org/models/resnet101-5d3b4d8f.pth"
target_path = "data/imagenet_weights/resnet101.pth"

if not os.path.exists(target_path):
    print("ResNet101 ImageNet 가중치 다운로드 중 (약 170MB)...")
    urllib.request.urlretrieve(resnet101_url, target_path)
    print("다운로드 완료!")
else:
    print("ResNet101 가중치 파일이 이미 존재합니다.")

ResNet101 가중치 파일이 이미 존재합니다.


In [3]:
# 현재 업로드한 두 파일이 최상위 경로에 올바르게 위치해 있는지 확인합니다.
required_files = ["model-best.pth", "infos_fc_nsc-best.pkl"]
all_clear = True

for f in required_files:
    if os.path.exists(f):
        print(f"[{f}] 파일 확인 완료.")
    else:
        print(f"[경고] {f} 파일이 최상위 경로에 없습니다. 위치를 확인해주세요.")
        all_clear = False

print("\n※ 추론을 진행하기 전에 'test_images' 폴더 안에 캡션을 생성할 이미지 파일(.jpg, .png 등)을 최소 1개 이상 넣어주세요.")

[model-best.pth] 파일 확인 완료.
[infos_fc_nsc-best.pkl] 파일 확인 완료.

※ 추론을 진행하기 전에 'test_images' 폴더 안에 캡션을 생성할 이미지 파일(.jpg, .png 등)을 최소 1개 이상 넣어주세요.


In [4]:
import pickle

original_pkl = 'infos_fc_nsc-best.pkl'
patched_pkl = 'infos_patched.pkl'

# 피클 파일 로드 (인코딩 에러 방지를 위해 encoding='latin1' 추가)
with open(original_pkl, 'rb') as f:
    infos = pickle.load(f, encoding='latin1')

# opt 속성 확인 및 'newfc'를 'fc'로 수정
if hasattr(infos['opt'], 'caption_model'):
    print(f"변경 전 모델명: {infos['opt'].caption_model}")
    infos['opt'].caption_model = 'fc'
    print(f"변경 후 모델명: {infos['opt'].caption_model}")
elif isinstance(infos['opt'], dict) and 'caption_model' in infos['opt']:
    print(f"변경 전 모델명: {infos['opt']['caption_model']}")
    infos['opt']['caption_model'] = 'fc'
    print(f"변경 후 모델명: {infos['opt']['caption_model']}")

# 패치된 파일 저장
with open(patched_pkl, 'wb') as f:
    pickle.dump(infos, f)

print(f"패치 완료: {patched_pkl} 파일이 정상적으로 생성되었습니다.")

변경 전 모델명: newfc
변경 후 모델명: fc
패치 완료: infos_patched.pkl 파일이 정상적으로 생성되었습니다.


In [5]:
import torch

original_pth = 'model-best.pth'
patched_pth = 'model_patched.pth'

# 1. 원본 가중치 로드 (메모리 문제 방지를 위해 cpu에 할당)
state_dict = torch.load(original_pth, map_location='cpu')

# 2. 새로운 레이어 이름을 담을 딕셔너리 생성
new_state_dict = {}

# 3. 에러 로그에 기반한 1:1 이름(Key) 매핑
for key, value in state_dict.items():
    new_key = key
    
    # fc_embed -> img_embed 로 변경
    if key.startswith('fc_embed.'):
        new_key = key.replace('fc_embed.', 'img_embed.')
    # _core -> core 로 변경
    elif key.startswith('_core.'):
        new_key = key.replace('_core.', 'core.')
        
    new_state_dict[new_key] = value
    
    # 변경된 내역을 출력하여 확인
    if key != new_key:
        print(f"변환: {key} -> {new_key}")

# 4. 이름이 수정된 새로운 가중치 파일 저장
torch.save(new_state_dict, patched_pth)
print(f"\n가중치 키 변환 완료! 새로운 파일이 저장되었습니다: {patched_pth}")

변환: fc_embed.weight -> img_embed.weight
변환: fc_embed.bias -> img_embed.bias
변환: _core.i2h.weight -> core.i2h.weight
변환: _core.i2h.bias -> core.i2h.bias
변환: _core.h2h.weight -> core.h2h.weight
변환: _core.h2h.bias -> core.h2h.bias

가중치 키 변환 완료! 새로운 파일이 저장되었습니다: model_patched.pth


In [6]:
import torch
import os

resnet_path = 'data/imagenet_weights/resnet101.pth'

if os.path.exists(resnet_path):
    print("구형 포맷의 resnet101 가중치를 불러오는 중 (보안 예외 허용)...")
    # weights_only=False 옵션을 명시하여 레거시 파일을 안전하게 로드합니다.
    legacy_weights = torch.load(resnet_path, weights_only=False)
    
    # 최신 PyTorch 기본 포맷으로 동일한 경로에 덮어쓰기 저장합니다.
    torch.save(legacy_weights, resnet_path)
    print("최신 포맷으로 변환 및 덮어쓰기 완료! 이제 호환성 에러가 발생하지 않습니다.")
else:
    print(f"파일을 찾을 수 없습니다: {resnet_path}")

구형 포맷의 resnet101 가중치를 불러오는 중 (보안 예외 허용)...
최신 포맷으로 변환 및 덮어쓰기 완료! 이제 호환성 에러가 발생하지 않습니다.


In [7]:
import os
os.environ['KMP_DUPLICATE_LIB_OK'] = 'True'

!python eval.py \
    --model model_patched.pth \
    --infos_path infos_patched.pkl \
    --image_folder test_images \
    --num_images -1 \
    --language_eval 0

DataLoaderRaw loading images from folder:  test_images
0
listing all images in directory test_images
DataLoaderRaw found  2  images
cp "test_images\dog.jpg" vis/imgs/img1.jpg
image 1: a dog is sitting on top of a book
cp "test_images\zebra.jpg" vis/imgs/img2.jpg
image 2: a dog is sitting on top of a table
cp "test_images\dog.jpg" vis/imgs/img3.jpg
image 1: a dog is sitting on top of a book
cp "test_images\zebra.jpg" vis/imgs/img4.jpg
image 2: a dog is sitting on top of a table
cp "test_images\dog.jpg" vis/imgs/img5.jpg
image 1: a dog is sitting on top of a book
cp "test_images\zebra.jpg" vis/imgs/img6.jpg
image 2: a dog is sitting on top of a table
cp "test_images\dog.jpg" vis/imgs/img7.jpg
image 1: a dog is sitting on top of a book
cp "test_images\zebra.jpg" vis/imgs/img8.jpg
image 2: a dog is sitting on top of a table
cp "test_images\dog.jpg" vis/imgs/img9.jpg
image 1: a dog is sitting on top of a book
cp "test_images\zebra.jpg" vis/imgs/img10.jpg
image 2: a dog is sitting on top of 

C:\Users\User\Desktop\python\딥러닝 2주차\test\models\FCModel.py:147: UserWarning: Implicit dimension choice for log_softmax has been deprecated. Change the call to include dim=X as an argument.
  logprobs = F.log_softmax(self.logit(output))
C:\Users\User\Desktop\python\딥러닝 2주차\test\models\FCModel.py:123: UserWarning: Implicit dimension choice for log_softmax has been deprecated. Change the call to include dim=X as an argument.
  logprobs = F.log_softmax(self.logit(output))
'cp'은(는) 내부 또는 외부 명령, 실행할 수 있는 프로그램, 또는
배치 파일이 아닙니다.
'cp'은(는) 내부 또는 외부 명령, 실행할 수 있는 프로그램, 또는
배치 파일이 아닙니다.
'cp'은(는) 내부 또는 외부 명령, 실행할 수 있는 프로그램, 또는
배치 파일이 아닙니다.
'cp'은(는) 내부 또는 외부 명령, 실행할 수 있는 프로그램, 또는
배치 파일이 아닙니다.
'cp'은(는) 내부 또는 외부 명령, 실행할 수 있는 프로그램, 또는
배치 파일이 아닙니다.
'cp'은(는) 내부 또는 외부 명령, 실행할 수 있는 프로그램, 또는
배치 파일이 아닙니다.
'cp'은(는) 내부 또는 외부 명령, 실행할 수 있는 프로그램, 또는
배치 파일이 아닙니다.
'cp'은(는) 내부 또는 외부 명령, 실행할 수 있는 프로그램, 또는
배치 파일이 아닙니다.
'cp'은(는) 내부 또는 외부 명령, 실행할 수 있는 프로그램, 또는
배치 파일이 아닙니다.
'cp'은(는) 내부 또는 외부 명령, 실행할 수 있는 프로그램, 또는
배치 파일이 아닙

In [9]:
import json
import matplotlib.pyplot as plt
from PIL import Image
import os

result_json_path = 'vis/vis.json'

if os.path.exists(result_json_path):
    with open(result_json_path, 'r', encoding='utf-8') as f:
        results = json.load(f)
    
    print(f"총 {len(results)}개의 이미지에 대한 캡션이 생성되었습니다.\n")
    

총 2개의 이미지에 대한 캡션이 생성되었습니다.



In [ ]:
'''
import json
import matplotlib.pyplot as plt
from PIL import Image
import os

result_json_path = 'vis/vis.json'

if os.path.exists(result_json_path):
    with open(result_json_path, 'r', encoding='utf-8') as f:
        results = json.load(f)
    
    print(f"총 {len(results)}개의 이미지에 대한 캡션이 생성되었습니다.\n")
    
    for item in results:
        file_name = item.get('file_name', '')
        caption = item.get('caption', '')
        
        image_path = os.path.join('test_images', file_name)
        if os.path.exists(image_path):
            img = Image.open(image_path)
            plt.figure(figsize=(6, 6))
            plt.imshow(img)
            plt.title(f"Predicted Caption:\n{caption}", fontsize=12, color='blue')
            plt.axis('off')
            plt.show()
        else:
            print(f"이미지 파일을 찾을 수 없습니다: {file_name}")
else:
    print("결과 파일(vis/vis.json)을 찾을 수 없습니다.")
'''